# Propylene Oxide (−) PECD — 2W, 20260820

Forward/backward asymmetry from the `PO (-) PECD 2W` dataset.

Three quarter-wave-plate settings:

| angle | helicity | file |
|-------|----------|------|
| +45°  | one circular    | `pos_003_+45.0000_c.h5` |
| −45°  | other circular  | `pos_001_-45.0000_c.h5` |
|   0°  | linear (control)| `pos_002_+0.0000_c.h5`  |

`py` is the light-propagation axis: forward = `py > 0`, backward = `py < 0`.

The antisymmetrized PECD removes apparatus asymmetries:

$$\mathrm{PECD} = 2\left(\frac{F_{+}-B_{+}}{F_{+}+B_{+}} - \frac{F_{-}-B_{-}}{F_{-}+B_{-}}\right)$$

In [ ]:
import itertools

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
files = {
    45.0: r"J:\ctgroup\Edward\DATA\VMI\20260820\PO (-) PECD 2W\pos_003_+45.0000_calibrated.h5",
    -45.0: r"J:\ctgroup\Edward\DATA\VMI\20260820\PO (-) PECD 2W\pos_001_-45.0000_calibrated.h5",
    0.0: r"J:\ctgroup\Edward\DATA\VMI\20260820\PO (-) PECD 2W\pos_002_+0.0000_calibrated.h5",
}

PR_MAX = 0.8
MQ_MAX = 65

data = pd.DataFrame()
for angle, path in files.items():
    d = pd.read_hdf(path)
    d["angle"] = angle
    data = pd.concat([data, d], ignore_index=True)

# Offset correction: the linear (0 deg) run carries no PECD and should sit at
# zero net momentum, so its per-axis median is the detector/field offset.
# Restrict to the 10 < m/q < 60 fragments so stray masses don't skew it.
# Subtract it from every run before computing pr.
# lin_mask = (data["angle"] == 0.0) & data["m/q"].between(10, 60)
# offset = data.loc[lin_mask, ["px", "py", "pz"]].median()
# print("offset (px, py, pz):", offset.to_dict())
# data[["px", "py", "pz"]] = data[["px", "py", "pz"]] - offset

data["pr"] = np.sqrt(data["px"] ** 2 + data["py"] ** 2 + data["pz"] ** 2)
data = data[(data["pr"] < PR_MAX) & (data["m/q"] < MQ_MAX)]

print(data.groupby("angle").size())
data.head()

In [ ]:
sp = data[data["angle"] == 45.0]
sm = data[data["angle"] == -45.0]
s0 = data[data["angle"] == 0.0]


def fba(df):
    """Forward-backward asymmetry (F - B) / (F + B) along py."""
    f = int((df["py"] > 0).sum())
    b = int((df["py"] < 0).sum())
    n = f + b
    if n == 0:
        return np.nan, np.nan, 0
    a = (f - b) / n
    err = np.sqrt((1 - a ** 2) / n)  # binomial error on the asymmetry
    return a, err, n


def pecd_in_bins(col, edges, min_counts=1500):
    """Antisymmetrized PECD (+45 vs -45) and raw FBA per bin of `col`."""
    rows = []
    for lo, hi in itertools.pairwise(edges):
        pb = sp[(sp[col] > lo) & (sp[col] <= hi)]
        mb = sm[(sm[col] > lo) & (sm[col] <= hi)]
        zb = s0[(s0[col] > lo) & (s0[col] <= hi)]
        ap, ep, np_ = fba(pb)
        am, em, nm = fba(mb)
        a0, e0, n0 = fba(zb)
        n = np_ + nm
        pecd = 2 * (ap - am) if n >= min_counts else np.nan
        pecd_err = 2 * np.sqrt(ep ** 2 + em ** 2) if n >= min_counts else np.nan
        rows.append(
            dict(
                center=0.5 * (lo + hi),
                pecd=pecd,
                pecd_err=pecd_err,
                fba_plus=ap,
                fba_minus=am,
                fba_lin=a0,
                counts=n,
            )
        )
    return pd.DataFrame(rows)

In [ ]:
# m/q spectrum, to see which fragments carry signal
px.histogram(
    data, x="m/q", color="angle", nbins=400, barmode="overlay", opacity=0.6,
    range_x=[0, MQ_MAX], title="m/q spectrum",
)

In [ ]:
ap, ep, _ = fba(sp)
am, em, _ = fba(sm)
a0, e0, _ = fba(s0)
print(f"FBA(+45) = {ap:+.4f} +/- {ep:.4f}")
print(f"FBA(-45) = {am:+.4f} +/- {em:.4f}")
print(f"FBA(  0) = {a0:+.4f} +/- {e0:.4f}   (linear control, should be ~0)")
print(f"PECD     = {2 * (ap - am):+.4f} +/- {2 * np.sqrt(ep ** 2 + em ** 2):.4f}")

## Forward-backward asymmetry vs radial momentum

In [ ]:
pr_df = pecd_in_bins("pr", np.linspace(0, PR_MAX, 50), min_counts=1500)
pr_df

In [ ]:
fig = go.Figure(
    data=[
        go.Bar(
            x=pr_df["center"], y=pr_df["counts"], name="Signal", yaxis="y2",
            marker_color="lightgray", opacity=0.5,
        ),
        go.Scatter(
            x=pr_df["center"], y=pr_df["pecd"],
            error_y=dict(type="data", array=pr_df["pecd_err"]),
            name="PECD (antisym.)", mode="lines+markers", marker_color="blue",
        ),
        go.Scatter(
            x=pr_df["center"], y=pr_df["fba_plus"], name="FBA (+45°)",
            mode="lines", line=dict(color="red", dash="dot"),
        ),
        go.Scatter(
            x=pr_df["center"], y=pr_df["fba_minus"], name="FBA (−45°)",
            mode="lines", line=dict(color="green", dash="dot"),
        ),
        go.Scatter(
            x=pr_df["center"], y=pr_df["fba_lin"], name="FBA (0°)",
            mode="lines", line=dict(color="gray", dash="dash"),
        ),
    ],
    layout=go.Layout(
        title="Forward-backward asymmetry vs radial momentum",
        xaxis=dict(title="Radial momentum p_r (a.u.)"),
        yaxis=dict(title="Asymmetry", range=[-0.3, 0.3], showgrid=True, zeroline=True),
        yaxis2=dict(title="Signal (counts)", overlaying="y", side="right", showgrid=False),
        legend=dict(x=0.02, y=0.98),
        height=600, width=850, template="simple_white+presentation",
    ),
)
fig.add_hline(y=0, line_color="black", line_width=1)
fig.show()

## Forward-backward asymmetry vs m/q

In [ ]:
mq_df = pecd_in_bins("m/q", np.linspace(0, MQ_MAX, 50), min_counts=2000)
mq_df

In [ ]:
fig = go.Figure(
    data=[
        go.Bar(
            x=mq_df["center"], y=mq_df["counts"], name="Signal", yaxis="y2",
            marker_color="lightgray", opacity=0.5,
        ),
        go.Scatter(
            x=mq_df["center"], y=mq_df["pecd"],
            error_y=dict(type="data", array=mq_df["pecd_err"]),
            name="PECD (antisym.)", mode="lines+markers", marker_color="blue",
        ),
        go.Scatter(
            x=mq_df["center"], y=mq_df["fba_lin"], name="FBA (0°)",
            mode="lines", line=dict(color="lightgray", dash="dash"),
        ),
    ],
    layout=go.Layout(
        title="Forward-backward asymmetry vs m/q",
        xaxis=dict(title="m/q", range=[0, MQ_MAX]),
        yaxis=dict(title="Asymmetry", range=[-0.3, 0.3], showgrid=True, zeroline=True),
        yaxis2=dict(title="Signal (counts)", overlaying="y", side="right", showgrid=False),
        legend=dict(x=0.02, y=0.98),
        height=600, width=850, template="simple_white+presentation",
    ),
)
fig.add_hline(y=0, line_color="black", line_width=1)
fig.show()

## px–pz momentum maps for the top fragments (|p_y| < 0.05)

The four strongest fragments in the m/q spectrum (each a run of adjacent
m/q bins above threshold). One row per fragment, plus a combined "total"
row on top; columns are the three QWP settings (−45°, 0°, +45°). All maps
are the central slice `|py| < 0.05 a.u.`

In [ ]:
from plotly.subplots import make_subplots

PY_SLICE = 0.05      # keep |py| < PY_SLICE  (a.u.)
MQ_BIN_W = 0.25      # m/q histogram bin width for peak finding
PEAK_FRAC = 0.05     # a bin counts as signal if > PEAK_FRAC * tallest bin
N_PEAKS = 3          # keep the N strongest fragments
GRID_BINS = 120      # px/pz heatmap resolution
ANGLES = [-45.0, 0.0, 45.0]

# --- fragments = continuously connected m/q bins above threshold ---
mq_edges = np.arange(0, MQ_MAX + MQ_BIN_W, MQ_BIN_W)
mq_counts, _ = np.histogram(data["m/q"], bins=mq_edges)
above = mq_counts > PEAK_FRAC * mq_counts.max()

fragments = []  # (lo, hi, centroid, n_ions)
i = 0
while i < len(above):
    if not above[i]:
        i += 1
        continue
    j = i
    while j < len(above) and above[j]:
        j += 1
    lo, hi = mq_edges[i], mq_edges[j]
    m = data["m/q"].between(lo, hi)
    fragments.append((lo, hi, data.loc[m, "m/q"].mean(), int(m.sum())))
    i = j

# keep the N strongest, then order by m/q for display
fragments = sorted(sorted(fragments, key=lambda f: -f[3])[:N_PEAKS],
                   key=lambda f: f[2])

print(f"top {len(fragments)} fragments (|py| < {PY_SLICE} a.u.):")
for lo, hi, c, n in fragments:
    print(f"  m/q {lo:5.2f}-{hi:5.2f}  centroid {c:6.2f}  {n:>7d} ions")

# --- px-pz heatmaps in the central py slice ---
sl = data[data["py"].abs() < PY_SLICE]
edges = np.linspace(-PR_MAX, PR_MAX, GRID_BINS + 1)
centers = 0.5 * (edges[:-1] + edges[1:])

in_frag = np.zeros(len(sl), dtype=bool)
for lo, hi, c, n in fragments:
    in_frag |= sl["m/q"].between(lo, hi).to_numpy()

rows = [("total", in_frag)] + [
    (f"m/q {c:.1f}", sl["m/q"].between(lo, hi).to_numpy())
    for lo, hi, c, n in fragments
]

nrows = len(rows)
fig = make_subplots(
    rows=nrows, cols=len(ANGLES),
    row_titles=[r[0] for r in rows],
    column_titles=[f"{a:+.0f}°" for a in ANGLES],
    horizontal_spacing=0.05, vertical_spacing=0.04,
)
k = 0
for r, (label, mq_sel) in enumerate(rows):
    for cc, angle in enumerate(ANGLES):
        sub = sl[mq_sel & (sl["angle"] == angle).to_numpy()]
        h, _, _ = np.histogram2d(sub["px"], sub["pz"], bins=[edges, edges])
        fig.add_trace(
            go.Heatmap(z=h.T, x=centers, y=centers, colorscale="Inferno",
                       showscale=False),
            row=r + 1, col=cc + 1,
        )
        ax = "" if k == 0 else str(k + 1)
        fig.update_yaxes(scaleanchor=f"x{ax}", constrain="domain",
                         row=r + 1, col=cc + 1)
        k += 1

fig.update_layout(
    title=f"px-pz momentum, |py| < {PY_SLICE} a.u.",
    height=300 * nrows, width=300 * len(ANGLES),
    template="simple_white",
)
fig.show()